In [2]:
!pip install iterative-stratification

In [3]:
import os
import json
import numpy as np
import pandas as pd

import shutil

import matplotlib.pyplot as plt

import torch
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    f1_score
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

In [ ]:
import sys
import importlib.metadata

print(f"Python: {sys.version.split()[0]}")

packages = [
    "torch",
    "transformers",
    "scikit-learn",
    "pandas",
    "numpy",
    "iterative-stratification"
]

for pkg in packages:
    try:
        print(f"{pkg}: {importlib.metadata.version(pkg)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg}: Not installed")

In [4]:
SOURCE_FILE_NAME = "annotations.json"
FILTERED_FILE_NAME = "annotations_filtered.json"

ACD_EPOCHS_CSV = "acd_epochs.csv"
ACSA_EPOCHS_CSV = "acsa_epochs.csv"

# Diagram file paths
VIS_DATA_FILE_NAME = "token_lengths.json"
TOKEN_LENGTH_DISTRIBUTION_DIAGRAM_FILE_NAME = "token_length_distribution.png"

ACD_METRICS_PER_EPOCH = "acd_metrics_curves.png"
ACSA_METRICS_PER_EPOCH = "acsa_metrics_curves.png"

# Configurable Model Identifier: "classla/bcms-bertic" or "xlm-roberta-base"
MODEL_NAME = "xlm-roberta-base"
SEED = 42
MAX_LEN = 512
EPOCHS = 10
SAVE_EPOCHS = 2
LR = 2e-5
WARMUP_STEPS = 0.1
WEIGHT_DECAY = 0.01
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16


# Paths to save trained models
SAVE_DIR = "saved_models"
SAVE_DIR_ACD = f"./{SAVE_DIR}/acd"
SAVE_DIR_ACSA = f"./{SAVE_DIR}/acsa"

ALL_CATEGORIES = [
    "Baterija", "Kamera", "Ekran", "Memorija", "Zvučnici",
    "Izgled", "Hardver", "Softver", "Performanse", "Cena", "Opšta ocena"
]

POLARITIES_MAP = {
    "Neutralan": 0, "Pozitivan": 1, "Negativan": 2, "Konflikt": 3
}

INV_POLARITIES_MAP = {
    val: key for key, val in POLARITIES_MAP.items()
}

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

with open(SOURCE_FILE_NAME, "r", encoding="utf-8") as f:
    data = json.load(f)

total_reviews = 0
over_512_count = 0
max_tokens_found = 0
all_lengths = []

print("Starting comment analysis...\n")

for item in data:
    text = item["comment"]
    review_status = item["review_status"]

    if review_status == "NE":
        continue

    total_reviews += 1

    token_ids = tokenizer.encode(text, add_special_tokens=True, truncation=False)
    num_tokens = len(token_ids)

    all_lengths.append(num_tokens)

    if num_tokens > 512:
        over_512_count += 1
        item["review_status"] = "NE"

    if num_tokens > max_tokens_found:
        max_tokens_found = num_tokens

percent = (over_512_count / total_reviews) * 100
mean_length = sum(all_lengths) / total_reviews

print("=== RESULT OF ANALYSIS OF REVIEWS TOKEN LENGTH ===")
print(f"Total number of reviews: {total_reviews}")
print(f"Number of reviews that go OVER 512 tokens: {over_512_count} ({percent:.2f}%)")
print(f"Mean review token length: {mean_length:.1f} tokena")
print(f"Longest reviews has: {max_tokens_found} tokens")

with open(FILTERED_FILE_NAME, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4, ensure_ascii=False)

viz_data = {
    "total_reviews": total_reviews,
    "over_512_count": over_512_count,
    "mean_length": mean_length,
    "max_length": max_tokens_found,
    "all_lengths": all_lengths
}

with open(VIS_DATA_FILE_NAME, "w", encoding="utf-8") as f:
    json.dump(viz_data, f, indent=4, ensure_ascii=False)

print(f"\nSuccessfully saved visualization data to '{VIS_DATA_FILE_NAME}'.")

plt.figure(figsize=(10, 6))
plt.hist(all_lengths, bins=40, color='skyblue', edgecolor='black', alpha=0.7)
plt.axvline(x=512, color='red', linestyle='--', linewidth=2, label='Token limit (512)')

plt.title('Distribution of token length in reviews', fontsize=14)
plt.xlabel('No. tokens', fontsize=12)
plt.ylabel('No. reviews', fontsize=12)
plt.legend(fontsize=11)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()

plt.savefig(TOKEN_LENGTH_DISTRIBUTION_DIAGRAM_FILE_NAME, dpi=300)
plt.close()

print(f"Successfully saved diagram to '{TOKEN_LENGTH_DISTRIBUTION_DIAGRAM_FILE_NAME}'.")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (536 > 512). Running this sequence through the model will result in indexing errors


Starting comment analysis...

=== RESULT OF ANALYSIS OF REVIEWS TOKEN LENGTH ===
Total number of reviews: 6454
Number of reviews that go OVER 512 tokens: 19 (0.29%)
Mean review token length: 71.7 tokena
Longest reviews has: 1262 tokens

Successfully saved visualization data to 'token_lengths.json'.
Successfully saved diagram to 'token_length_distribution.png'.


In [14]:
# ===========================================
# 1. DATA PREPARATION (Document-Level Splitting)
# ===========================================
def load_and_split_data(json_file_path, seed=SEED):
    with open(json_file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Filter unreviewed / invalid reviews
    valid_data = [item for item in data if item.get("review_status") != "NE"]

    # Build Category::Polarity multi-label matrix across all combinations
    pair_labels = [
        f"{cat}::{pol}"
        for cat in ALL_CATEGORIES
        for pol in POLARITIES_MAP.keys()
    ]
    pair_to_idx = {pair: idx for idx, pair in enumerate(pair_labels)}

    matrix = np.zeros((len(valid_data), len(pair_labels)), dtype=np.uint8)
    for i, item in enumerate(valid_data):
        for aspect_category in item.get("aspect_categories", []):
            cat = aspect_category.get("category")
            pol = aspect_category.get("polarity")
            pair = f"{cat}::{pol}"
            if pair in pair_to_idx:
                matrix[i, pair_to_idx[pair]] = 1

    # Stage 1 Split: 80% Train, 20% Held-out
    split1 = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=0.20, random_state=seed
    )
    dummy_x = np.zeros((len(valid_data), 1), dtype=np.uint8)
    train_indices, held_out_indices = next(split1.split(dummy_x, matrix))

    # Stage 2 Split: Split Held-out 50/50 into Val and Test (10% and 10% global)
    split2 = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=0.50, random_state=seed + 1
    )
    held_out_matrix = matrix[held_out_indices]
    dummy_held_out_x = np.zeros((len(held_out_matrix), 1), dtype=np.uint8)
    val_local, test_local = next(split2.split(dummy_held_out_x, held_out_matrix))

    val_indices = held_out_indices[val_local]
    test_indices = held_out_indices[test_local]

    train_items = [valid_data[i] for i in train_indices]
    val_items = [valid_data[i] for i in val_indices]
    test_items = [valid_data[i] for i in test_indices]

    return train_items, val_items, test_items

def extract_absa_samples(items):
    acd_comments, acd_labels = [], []
    acsa_comments, acsa_categories, acsa_labels = [], [], []
    gold_tuples = []  # Set of (category, polarity_id) per document

    for item in items:
        comment = item["comment"]
        aspect_categories = item.get("aspect_categories", [])

        # ACD binary indicator vector
        acd_comments.append(comment)
        label_vector = [0.0] * len(ALL_CATEGORIES)
        doc_tuples = set()

        for aspect_category in aspect_categories:
            category = aspect_category.get("category")
            polarity = aspect_category.get("polarity")

            if category in ALL_CATEGORIES:
                idx = ALL_CATEGORIES.index(category)
                label_vector[idx] = 1.0

                # ACSA Comment + Category pair
                if polarity in POLARITIES_MAP:
                    pol_id = POLARITIES_MAP[polarity]
                    acsa_comments.append(comment)
                    acsa_categories.append(category)
                    acsa_labels.append(pol_id)
                    doc_tuples.add((category, pol_id))

        acd_labels.append(label_vector)
        gold_tuples.append(doc_tuples)

    return (acd_comments, acd_labels), (acsa_comments, acsa_categories, acsa_labels), gold_tuples

def print_split_statistics(train_items, val_items, test_items):
    splits = {"Train": train_items, "Val": val_items, "Test": test_items}
    total_docs = sum(len(items) for items in splits.values())

    print("\n" + "=" * 70)
    print(" 1. DOCUMENT-LEVEL SPLIT RATIOS")
    print("=" * 70)
    for name, items in splits.items():
        count = len(items)
        pct = (count / total_docs) * 100
        print(f" {name:<10}: {count:>5} comments ({pct:>6.2f}%)")

    # Aggregate counts across splits
    pair_labels = [
        f"{cat}::{pol}"
        for cat in ALL_CATEGORIES
        for pol in POLARITIES_MAP.keys()
    ]
    pair_stats = {name: {pair: 0 for pair in pair_labels} for name in splits}
    cat_stats = {name: {cat: 0 for cat in ALL_CATEGORIES} for name in splits}

    for name, items in splits.items():
        for item in items:
            for aspect in item.get("aspect_categories", []):
                cat = aspect.get("category")
                pol = aspect.get("polarity")
                pair = f"{cat}::{pol}"
                if pair in pair_stats[name]:
                    pair_stats[name][pair] += 1
                if cat in cat_stats[name]:
                    cat_stats[name][cat] += 1

    print("\n" + "=" * 70)
    print(" 2. CATEGORY-LEVEL DISTRIBUTION (ACD)")
    print(" Target split ratio: Train (~80%) | Val (~10%) | Test (~10%)")
    print("=" * 70)
    print(f" {'Category':<15} | {'Total':<6} | {'Train %':<8} | {'Val %':<8} | {'Test %':<8}")
    print("-" * 70)
    for cat in ALL_CATEGORIES:
        tr_c = cat_stats["Train"][cat]
        va_c = cat_stats["Val"][cat]
        te_c = cat_stats["Test"][cat]
        tot = tr_c + va_c + te_c
        if tot > 0:
            print(f" {cat:<15} | {tot:<6} | {tr_c/tot:>7.1%} | {va_c/tot:>7.1%} | {te_c/tot:>7.1%}")
        else:
            print(f" {cat:<15} | {tot:<6} | {'N/A':>7} | {'N/A':>7} | {'N/A':>7}")

    print("\n" + "=" * 70)
    print(" 3. CATEGORY::POLARITY PAIR DISTRIBUTION (ACSA)")
    print("=" * 70)
    print(f" {'Pair Label':<30} | {'Total':<6} | {'Train %':<8} | {'Val %':<8} | {'Test %':<8}")
    print("-" * 70)

    # Sort pairs by frequency
    sorted_pairs = sorted(
        pair_labels,
        key=lambda p: sum(pair_stats[n][p] for n in splits),
        reverse=True
    )

    for pair in sorted_pairs:
        tr_c = pair_stats["Train"][pair]
        va_c = pair_stats["Val"][pair]
        te_c = pair_stats["Test"][pair]
        tot = tr_c + va_c + te_c
        if tot > 0:
            print(f" {pair:<30} | {tot:<6} | {tr_c/tot:>7.1%} | {va_c/tot:>7.1%} | {te_c/tot:>7.1%}")
    print("=" * 70 + "\n")

In [7]:
# ===========================================
# 2. LAZY DATASETS (Dynamic Padding)
# ===========================================
class LazyACDDataset(torch.utils.data.Dataset):
    def __init__(self, comments, labels, tokenizer, max_len=MAX_LEN):
        self.comments = comments
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        item = self.tokenizer(
            self.comments[idx],
            truncation=True,
            max_length=self.max_len
        )
        item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

class LazyACSADataset(torch.utils.data.Dataset):
    def __init__(self, comments, categories, labels, tokenizer, max_len=MAX_LEN):
        self.comments = comments
        self.categories = categories
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        item = self.tokenizer(
            self.comments[idx],
            text_pair=self.categories[idx],
            truncation=True,
            max_length=self.max_len
        )
        item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

In [8]:
# ===========================================
# 3. METRICS & THRESHOLD OPTIMIZATION
# ===========================================
def get_compute_metrics_acd(threshold=0.5):
    def compute_metrics_acd(eval_pred):
        logits, labels = eval_pred
        probs = 1 / (1 + np.exp(-logits))
        predictions = (probs > threshold).astype(int)
        acc = accuracy_score(labels, predictions)
        macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(labels, predictions, average="macro", zero_division=0)
        micro_f1 = f1_score(labels, predictions, average="micro", zero_division=0)
        weighted_f1 = f1_score(labels, predictions, average="weighted", zero_division=0)
        return {
            "acd_accuracy": acc,
            "acd_macro_f1": macro_f1,
            "acd_macro_precision": macro_precision,
            "acd_macro_recall": macro_recall,
            "acd_micro_f1": micro_f1,
            "acd_weighted_f1": weighted_f1
        }
    return compute_metrics_acd

def compute_metrics_acsa(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(labels, predictions, average="macro", zero_division=0)
    micro_f1 = f1_score(labels, predictions, average="micro", zero_division=0)
    weighted_f1 = f1_score(labels, predictions, average="weighted", zero_division=0)
    return {
        "acsa_accuracy": acc,
        "acsa_macro_f1": macro_f1,
        "acsa_macro_precision": macro_precision,
        "acsa_macro_recall": macro_recall,
        "acsa_micro_f1": micro_f1,
        "acsa_weighted_f1": weighted_f1
    }

def find_best_acd_thresholds_per_class(
    model, val_dataset, data_collator, device, categories
):
    model.to(device).eval()
    val_loader = torch.utils.data.DataLoader(
        val_dataset, batch_size=16, collate_fn=data_collator
    )

    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            labels = batch.pop("labels")
            inputs = {k: v.to(device) for k, v in batch.items()}
            logits = model(**inputs).logits
            all_logits.append(logits.cpu().numpy())
            all_labels.append(labels.numpy())

    all_logits = np.vstack(all_logits)
    all_labels = np.vstack(all_labels)
    probs = 1 / (1 + np.exp(-all_logits))

    best_thresholds = {}
    grid = np.arange(0.05, 0.90, 0.05)

    print("\n[Validation] Optimizing Per-Class ACD Thresholds:")
    for idx, cat in enumerate(categories):
        cat_probs = probs[:, idx]
        cat_labels = all_labels[:, idx]

        best_t = 0.5
        best_f1 = -1.0

        for thresh in grid:
            preds = (cat_probs > thresh).astype(int)
            f1 = f1_score(cat_labels, preds, average="binary", zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = thresh

        best_thresholds[cat] = float(best_t)
        print(
            f" - {cat:<15}: Threshold = {best_t:.2f} (Val F1: {best_f1:.4f})"
        )

    return best_thresholds

def evaluate_acd_overall(
    model, dataset, data_collator, thresholds, categories, device
):
    model.to(device).eval()
    loader = torch.utils.data.DataLoader(
        dataset, batch_size=16, collate_fn=data_collator
    )

    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop("labels")
            inputs = {k: v.to(device) for k, v in batch.items()}
            logits = model(**inputs).logits
            all_logits.append(logits.cpu().numpy())
            all_labels.append(labels.numpy())

    all_logits = np.vstack(all_logits)
    all_labels = np.vstack(all_labels)
    probs = 1 / (1 + np.exp(-all_logits))

    if isinstance(thresholds, dict):
        thresh_arr = np.array([thresholds[cat] for cat in categories])
    else:
        thresh_arr = np.array(thresholds)

    predictions = (probs > thresh_arr).astype(int)

    acc = accuracy_score(all_labels, predictions)
    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            all_labels, predictions, average="macro", zero_division=0
        )
    )
    micro_f1 = f1_score(
        all_labels, predictions, average="micro", zero_division=0
    )
    weighted_f1 = f1_score(
        all_labels, predictions, average="weighted", zero_division=0
    )

    precisions, recalls, f1s, supports = precision_recall_fscore_support(
        all_labels, predictions, average=None, zero_division=0
    )

    category_metrics = []
    for idx, cat in enumerate(categories):
        category_metrics.append(
            {
                "category": cat,
                "threshold": float(thresh_arr[idx]),
                "f1": f1s[idx],
                "precision": precisions[idx],
                "recall": recalls[idx],
                "support": int(supports[idx]),
            }
        )

    category_metrics.sort(key=lambda x: x["f1"], reverse=True)

    return {
        "acd_accuracy": acc,
        "acd_macro_f1": macro_f1,
        "acd_macro_precision": macro_precision,
        "acd_macro_recall": macro_recall,
        "acd_micro_f1": micro_f1,
        "acd_weighted_f1": weighted_f1,
        "category_metrics": category_metrics
    }

In [9]:
# ===========================================
# 4. END-TO-END PIPELINE EVALUATION
# ===========================================
def evaluate_end_to_end(
    acd_model, acd_test_ds, acsa_model, tokenizer, test_comments, gold_tuples_list, acd_thresholds, batch_size=16
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    acd_model.to(device).eval()
    acsa_model.to(device).eval()

    if isinstance(acd_thresholds, dict):
        thresh_arr = np.array([acd_thresholds[cat] for cat in ALL_CATEGORIES])
    else:
        thresh_arr = np.array(acd_thresholds)

    # Run ACD inference in batches
    acd_loader = torch.utils.data.DataLoader(
        acd_test_ds, batch_size=batch_size, collate_fn=DataCollatorWithPadding(tokenizer)
    )
    all_acd_probs = []
    with torch.no_grad():
        for batch in acd_loader:
            batch.pop("labels", None)
            inputs = {k: v.to(device) for k, v in batch.items()}
            logits = acd_model(**inputs).logits
            all_acd_probs.append(torch.sigmoid(logits).cpu().numpy())
    all_acd_probs = np.vstack(all_acd_probs)

    # Collect positive pairs & track indexing
    acsa_inputs, pair_indices = [], []
    y_true, y_pred = [], []
    active_labels = [0, 1, 2, 3]
    pred_tuples_list = [set() for _ in range(len(test_comments))]

    for doc_idx, (comment, gold_tuples) in enumerate(zip(test_comments, gold_tuples_list)):
        gold_dict = dict(gold_tuples)
        acd_probs = all_acd_probs[doc_idx]

        for cat_idx, cat in enumerate(ALL_CATEGORIES):
            gold_pol = gold_dict.get(cat, -1)
            y_true.append(gold_pol)
            curr_flat_idx = len(y_true) - 1

            if acd_probs[cat_idx] > thresh_arr[cat_idx]:
                acsa_inputs.append((comment, cat))
                pair_indices.append(curr_flat_idx)
                y_pred.append(-1)
            else:
                y_pred.append(-1)

    # Run ACSA inference
    if acsa_inputs:
        acsa_preds = []
        for i in range(0, len(acsa_inputs), batch_size):
            batch_pairs = acsa_inputs[i:i + batch_size]
            encoded = tokenizer(
                [p[0] for p in batch_pairs],
                [p[1] for p in batch_pairs],
                padding=True,
                truncation=True,
                max_length=MAX_LEN,
                return_tensors="pt"
            ).to(device)
            with torch.no_grad():
                logits = acsa_model(**encoded).logits
                acsa_preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())

        for idx, pred_pol in zip(pair_indices, acsa_preds):
            y_pred[idx] = pred_pol

    # Populate predicted tuple sets
    for flat_idx, pred_pol in enumerate(y_pred):
        if pred_pol in active_labels:
            doc_idx = flat_idx // len(ALL_CATEGORIES)
            cat = ALL_CATEGORIES[flat_idx % len(ALL_CATEGORIES)]
            pred_tuples_list[doc_idx].add((cat, pred_pol))

    # --- Existing Flat-Level Metrics ---
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=active_labels, average="macro", zero_division=0
    )
    micro_precision, micro_recall, micro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=active_labels, average="micro", zero_division=0
    )
    weighted_f1 = f1_score(
        y_true, y_pred, labels=active_labels, average="weighted", zero_division=0
    )

    # --- Tuple-Level Metrics ---
    total_tp, total_fp, total_fn = 0, 0, 0
    doc_f1s = []

    for gold_tuples, pred_tuples in zip(gold_tuples_list, pred_tuples_list):
        tp = len(gold_tuples & pred_tuples)
        fp = len(pred_tuples - gold_tuples)
        fn = len(gold_tuples - pred_tuples)

        total_tp += tp
        total_fp += fp
        total_fn += fn

        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        doc_f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        doc_f1s.append(doc_f1)

    tuple_micro_prec = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    tuple_micro_rec = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    tuple_micro_f1 = (2 * tuple_micro_prec * tuple_micro_rec) / (tuple_micro_prec + tuple_micro_rec) if (tuple_micro_prec + tuple_micro_rec) > 0 else 0.0

    return {
        "e2e_macro_f1": macro_f1,
        "e2e_macro_precision": macro_precision,
        "e2e_macro_recall": macro_recall,
        "e2e_micro_f1": micro_f1,
        "e2e_micro_precision": micro_precision,
        "e2e_micro_recall": micro_recall,
        "e2e_weighted_f1": weighted_f1,
        # tuple metrics
        "tuple_macro_f1": float(np.mean(doc_f1s)),
        "tuple_micro_f1": tuple_micro_f1,
        "tuple_micro_precision": tuple_micro_prec,
        "tuple_micro_recall": tuple_micro_rec,
    }

In [10]:
# ===========================================
# 5. SAVE PER EPOCH TRAINING METRICS
# ===========================================
def save_epoch_metrics_to_csv(log_history, csv_filename, task_type="acd"):
    eval_logs = [entry for entry in log_history if "eval_loss" in entry]
    train_logs = [entry for entry in log_history if "loss" in entry]

    rows = []
    for eval_entry in eval_logs:
        epoch_num = int(round(eval_entry["epoch"]))

        # Pick the latest logged training step loss at or before this evaluation
        matching_train = [t for t in train_logs if t["epoch"] <= eval_entry["epoch"]]
        train_loss = matching_train[-1]["loss"] if matching_train else eval_entry.get("loss", np.nan)

        prefix = task_type.lower()
        prefix_cap = prefix.capitalize()

        row = {
            "Epoch": epoch_num,
            "Training Loss": train_loss,
            "Validation Loss": eval_entry.get("eval_loss", np.nan),
            f"{prefix_cap} Accuracy": eval_entry.get(f"eval_{prefix}_accuracy", np.nan),
            f"{prefix_cap} Macro F1": eval_entry.get(f"eval_{prefix}_macro_f1", np.nan),
            f"{prefix_cap} Macro Precision": eval_entry.get(f"eval_{prefix}_macro_precision", np.nan),
            f"{prefix_cap} Macro Recall": eval_entry.get(f"eval_{prefix}_macro_recall", np.nan),
            f"{prefix_cap} Micro F1": eval_entry.get(f"eval_{prefix}_micro_f1", np.nan),
            f"{prefix_cap} Weighted F1": eval_entry.get(f"eval_{prefix}_weighted_f1", np.nan),
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    # Deduplicate evaluations for the same epoch, keeping the first (in-training) evaluation
    df = df.drop_duplicates(subset=["Epoch"], keep="first")

    df.to_csv(csv_filename, index=False, float_format="%.6f")
    print(f"Successfully generated and saved {csv_filename}")

In [11]:
# ===========================================
# 6. MAIN EXECUTION FLOW
# ===========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 1. Load and split data
train_items, val_items, test_items = load_and_split_data(FILTERED_FILE_NAME)

(acd_train_c, acd_train_l), (acsa_train_c, acsa_train_cat, acsa_train_l), _ = extract_absa_samples(train_items)
(acd_val_c, acd_val_l), (acsa_val_c, acsa_val_cat, acsa_val_l), _ = extract_absa_samples(val_items)
(acd_test_c, acd_test_l), (acsa_test_c, acsa_test_cat, acsa_test_l), gold_test_tuples = extract_absa_samples(test_items)

# 2. Build Datasets
acd_train_ds = LazyACDDataset(acd_train_c, acd_train_l, tokenizer)
acd_val_ds = LazyACDDataset(acd_val_c, acd_val_l, tokenizer)
acd_test_ds = LazyACDDataset(acd_test_c, acd_test_l, tokenizer)

acsa_train_ds = LazyACSADataset(acsa_train_c, acsa_train_cat, acsa_train_l, tokenizer)
acsa_val_ds = LazyACSADataset(acsa_val_c, acsa_val_cat, acsa_val_l, tokenizer)
acsa_test_ds = LazyACSADataset(acsa_test_c, acsa_test_cat, acsa_test_l, tokenizer)

In [15]:
print_split_statistics(train_items, val_items, test_items)


 1. DOCUMENT-LEVEL SPLIT RATIOS
 Train     :  5147 comments ( 79.98%)
 Val       :   660 comments ( 10.26%)
 Test      :   628 comments (  9.76%)

 2. CATEGORY-LEVEL DISTRIBUTION (ACD)
 Target split ratio: Train (~80%) | Val (~10%) | Test (~10%)
 Category        | Total  | Train %  | Val %    | Test %  
----------------------------------------------------------------------
 Baterija        | 2184   |   79.9% |   10.0% |   10.1%
 Kamera          | 1422   |   80.0% |   10.0% |   10.0%
 Ekran           | 956    |   80.0% |   10.0% |    9.9%
 Memorija        | 162    |   80.2% |    9.9% |    9.9%
 Zvučnici        | 497    |   80.1% |    9.7% |   10.3%
 Izgled          | 836    |   80.0% |   10.0% |    9.9%
 Hardver         | 1381   |   80.0% |   10.1% |    9.9%
 Softver         | 1838   |   80.0% |   10.0% |   10.0%
 Performanse     | 1123   |   80.0% |   10.0% |   10.1%
 Cena            | 853    |   80.0% |   10.2% |    9.8%
 Opšta ocena     | 3084   |   80.0% |   10.0% |   10.0%

 3. CA

In [13]:
# --- STAGE 1: TRAIN ACD ---
acd_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(ALL_CATEGORIES),
    problem_type="multi_label_classification"
)
acd_args = TrainingArguments(
    output_dir="./results_acd",
    num_train_epochs=EPOCHS,
    save_total_limit=SAVE_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="acd_macro_f1",
    greater_is_better=True,
    logging_steps=50
)
acd_trainer = Trainer(
    model=acd_model,
    args=acd_args,
    train_dataset=acd_train_ds,
    eval_dataset=acd_val_ds,
    compute_metrics=get_compute_metrics_acd(0.5),
    data_collator=data_collator,
    processing_class=tokenizer
)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
print("--- Starting Training for Stage 1: ACD ---")
acd_trainer.train()

--- Starting Training for Stage 1: ACD ---


Epoch,Training Loss,Validation Loss,Acd Accuracy,Acd Macro F1,Acd Macro Precision,Acd Macro Recall,Acd Micro F1,Acd Weighted F1
1,0.382778,0.342190,0.237879,0.357247,0.472731,0.335139,0.582631,0.488169
2,0.292596,0.273004,0.345455,0.571961,0.688357,0.534878,0.701835,0.655927
3,0.230391,0.221178,0.437879,0.658970,0.742725,0.618020,0.765060,0.741138
4,0.184624,0.197781,0.484848,0.722521,0.742075,0.711384,0.808855,0.800051
5,0.145038,0.183759,0.533333,0.749963,0.751378,0.750241,0.831187,0.824672
6,0.124338,0.184413,0.540909,0.764141,0.793975,0.751118,0.832803,0.825803
7,0.102645,0.177391,0.553030,0.796803,0.821091,0.778310,0.834103,0.832077
8,0.096434,0.177196,0.566667,0.828706,0.821343,0.837811,0.847083,0.846319
9,0.080480,0.174539,0.559091,0.824689,0.824596,0.825921,0.843826,0.843266
10,0.074057,0.174679,0.569697,0.831856,0.825083,0.839189,0.848464,0.848129


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3220, training_loss=0.187499558592435, metrics={'train_runtime': 1425.3186, 'train_samples_per_second': 36.111, 'train_steps_per_second': 2.259, 'total_flos': 5812730608606488.0, 'train_loss': 0.187499558592435, 'epoch': 10.0})

In [17]:
#log_df = pd.DataFrame(acd_trainer.state.log_history)
#print(log_df.head())

# Export Stage 1 metrics to CSV
save_epoch_metrics_to_csv(
    acd_trainer.state.log_history,
    ACD_EPOCHS_CSV,
    task_type="acd")

Successfully generated and saved acd_epochs.csv


In [18]:
# Find optimal per-class probability thresholds on validation set
best_acd_thresholds = find_best_acd_thresholds_per_class(
    acd_model, acd_val_ds, data_collator, device, ALL_CATEGORIES
)


[Validation] Optimizing Per-Class ACD Thresholds:
 - Baterija       : Threshold = 0.50 (Val F1: 0.9573)
 - Kamera         : Threshold = 0.15 (Val F1: 0.9139)
 - Ekran          : Threshold = 0.50 (Val F1: 0.8705)
 - Memorija       : Threshold = 0.50 (Val F1: 0.6667)
 - Zvučnici       : Threshold = 0.50 (Val F1: 0.9375)
 - Izgled         : Threshold = 0.55 (Val F1: 0.7826)
 - Hardver        : Threshold = 0.50 (Val F1: 0.8145)
 - Softver        : Threshold = 0.30 (Val F1: 0.8245)
 - Performanse    : Threshold = 0.40 (Val F1: 0.7745)
 - Cena           : Threshold = 0.30 (Val F1: 0.8298)
 - Opšta ocena    : Threshold = 0.80 (Val F1: 0.8398)


In [19]:
# Overall and per-category ACD evaluation on Test Set using per-class thresholds
acd_results = evaluate_acd_overall(
    acd_model,
    acd_test_ds,
    data_collator,
    best_acd_thresholds,
    ALL_CATEGORIES,
    device,
)
print("\n--- Standalone Overall Evaluation for ACD (Test Set) ---")
for metric, val in acd_results.items():
    if metric != "category_metrics":
        print(f"  {metric:<20}: {val:.4f}")

print("\n--- ACD Per-Category Evaluation (Test Set, Sorted by F1) ---")
print(
    f"{'Category':<15} | {'Thresh':<6} | {'F1 Score':<10} | {'Precision':<10} |"
    " {'Recall':<10} | {'Support':<8}"
)
print("-" * 75)
for item in acd_results["category_metrics"]:
    print(
        f"{item['category']:<15} | {item['threshold']:<6.2f} |"
        f" {item['f1']:<10.4f} | {item['precision']:<10.4f} |"
        f" {item['recall']:<10.4f} | {item['support']:<8}"
    )


--- Standalone Overall Evaluation for ACD (Test Set) ---
  acd_accuracy        : 0.5334
  acd_macro_f1        : 0.7935
  acd_macro_precision : 0.8122
  acd_macro_recall    : 0.7967
  acd_micro_f1        : 0.8391
  acd_weighted_f1     : 0.8375

--- ACD Per-Category Evaluation (Test Set, Sorted by F1) ---
Category        | Thresh | F1 Score   | Precision  | {'Recall':<10} | {'Support':<8}
---------------------------------------------------------------------------
Baterija        | 0.50   | 0.9422     | 0.9217     | 0.9636     | 220     
Zvučnici        | 0.50   | 0.9184     | 0.9574     | 0.8824     | 51      
Ekran           | 0.50   | 0.9026     | 0.8800     | 0.9263     | 95      
Kamera          | 0.15   | 0.8997     | 0.8323     | 0.9789     | 142     
Opšta ocena     | 0.80   | 0.8591     | 0.9004     | 0.8214     | 308     
Softver         | 0.30   | 0.7757     | 0.7500     | 0.8033     | 183     
Cena            | 0.30   | 0.7742     | 0.7059     | 0.8571     | 84      
Izgled  

In [20]:
# --- SAVE STAGE 1 (ACD MODEL & CONFIG) ---
print(f"\n[Saving] Saving Stage 1 (ACD) model to {SAVE_DIR_ACD}...")
acd_trainer.save_model(SAVE_DIR_ACD)
tokenizer.save_pretrained(SAVE_DIR_ACD)

acd_config = {
    "best_thresholds": best_acd_thresholds,
    "categories": ALL_CATEGORIES,
}
with open(os.path.join(SAVE_DIR_ACD, "acd_config.json"), "w", encoding="utf-8") as f:
    json.dump(acd_config, f, ensure_ascii=False, indent=2)


[Saving] Saving Stage 1 (ACD) model to ./saved_models/acd...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [21]:
# --- STAGE 2: TRAIN ACSA ---
acsa_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(POLARITIES_MAP)
)
acsa_args = TrainingArguments(
    output_dir="./results_acsa",
    num_train_epochs=EPOCHS,
    save_total_limit=SAVE_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="acsa_macro_f1",
    greater_is_better=True,
    logging_steps=50
)
acsa_trainer = Trainer(
    model=acsa_model,
    args=acsa_args,
    train_dataset=acsa_train_ds,
    eval_dataset=acsa_val_ds,
    compute_metrics=compute_metrics_acsa,
    data_collator=data_collator,
    processing_class=tokenizer
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
print("\n--- Starting Training for Stage 2: ACSA ---")
acsa_trainer.train()


--- Starting Training for Stage 2: ACSA ---


Epoch,Training Loss,Validation Loss,Acsa Accuracy,Acsa Macro F1,Acsa Macro Precision,Acsa Macro Recall,Acsa Micro F1,Acsa Weighted F1
1,0.644174,0.603354,0.802091,0.412991,0.400371,0.426537,0.802091,0.775351
2,0.480614,0.558663,0.834146,0.429522,0.420506,0.441271,0.834146,0.805862
3,0.445665,0.556694,0.822997,0.433892,0.461865,0.442614,0.822997,0.797557
4,0.332411,0.561080,0.842509,0.523510,0.605671,0.501378,0.842509,0.826909
5,0.323760,0.599207,0.839024,0.566843,0.606009,0.552948,0.839024,0.834126
6,0.221859,0.680025,0.827875,0.568816,0.573252,0.582410,0.827875,0.834111
7,0.202751,0.724517,0.848084,0.583674,0.608925,0.567283,0.848084,0.842096
8,0.175166,0.853324,0.824390,0.569290,0.561124,0.581668,0.824390,0.828808
9,0.144440,0.914586,0.836237,0.585046,0.578171,0.602634,0.836237,0.839375
10,0.117986,0.927671,0.834146,0.577604,0.573770,0.585225,0.834146,0.836501


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7170, training_loss=0.34501205860654016, metrics={'train_runtime': 3598.99, 'train_samples_per_second': 31.867, 'train_steps_per_second': 1.992, 'total_flos': 1.6534880133379296e+16, 'train_loss': 0.34501205860654016, 'epoch': 10.0})

In [23]:
# Export Stage 2 metrics to CSV
save_epoch_metrics_to_csv(
    acsa_trainer.state.log_history,
    ACSA_EPOCHS_CSV,
    task_type="acsa")

Successfully generated and saved acsa_epochs.csv


In [24]:
# Standalone ACSA (Oracle) Test Evaluation
print("\n--- Standalone Oracle Evaluation for ACSA (Test Set) ---")
acsa_test_results = acsa_trainer.evaluate(eval_dataset=acsa_test_ds)
print(acsa_test_results)


--- Standalone Oracle Evaluation for ACSA (Test Set) ---


Training Loss,Validation Loss,Epoch,Acsa Accuracy,Acsa Macro F1,Acsa Macro Precision,Acsa Macro Recall,Acsa Micro F1,Acsa Weighted F1
0.117986,0.891788,10,0.842877,0.594274,0.612116,0.587369,0.842877,0.844165


{'eval_loss': 0.8917882442474365, 'eval_acsa_accuracy': 0.8428770949720671, 'eval_acsa_macro_f1': 0.5942744820667284, 'eval_acsa_macro_precision': 0.6121155301948238, 'eval_acsa_macro_recall': 0.5873694828325616, 'eval_acsa_micro_f1': 0.8428770949720671, 'eval_acsa_weighted_f1': 0.8441652882717146}


In [25]:
# --- SAVE STAGE 2 (ACSA MODEL & CONFIG) ---
print(f"\n[Saving] Saving Stage 2 (ACSA) model to {SAVE_DIR_ACSA}...")
acsa_trainer.save_model(SAVE_DIR_ACSA)
tokenizer.save_pretrained(SAVE_DIR_ACSA)

acsa_config = {
    "polarities_map": POLARITIES_MAP
}
with open(os.path.join(SAVE_DIR_ACSA, "acsa_config.json"), "w", encoding="utf-8") as f:
    json.dump(acsa_config, f, ensure_ascii=False, indent=2)


[Saving] Saving Stage 2 (ACSA) model to ./saved_models/acsa...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [26]:
# --- END-TO-END PIPELINE EVALUATION ---
print("\n--- End-to-End Pipeline Evaluation (Test Set) ---")
e2e_results = evaluate_end_to_end(
    acd_model=acd_model,
    acd_test_ds=acd_test_ds,
    acsa_model=acsa_model,
    tokenizer=tokenizer,
    test_comments=acd_test_c,
    gold_tuples_list=gold_test_tuples,
    acd_thresholds=best_acd_thresholds,
)
print(e2e_results)


--- End-to-End Pipeline Evaluation (Test Set) ---
{'e2e_macro_f1': 0.5066914566063115, 'e2e_macro_precision': 0.5083484078740704, 'e2e_macro_recall': 0.5128623927004715, 'e2e_micro_f1': 0.7085635359116023, 'e2e_micro_precision': 0.7008196721311475, 'e2e_micro_recall': 0.7164804469273743, 'e2e_weighted_f1': 0.7119699867356007, 'tuple_macro_f1': 0.7014580518561411, 'tuple_micro_f1': 0.7085635359116021, 'tuple_micro_precision': 0.7008196721311475, 'tuple_micro_recall': 0.7164804469273743}


In [27]:
# Creates a f'{SAVE_DIR}.zip' file in your current directory
shutil.make_archive(SAVE_DIR, "zip", SAVE_DIR)

'/content/saved_models.zip'

In [28]:
df_acd = pd.read_csv(ACD_EPOCHS_CSV)
df_acsa = pd.read_csv(ACSA_EPOCHS_CSV)

# Setup plot style
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
epochs = range(1, 11)

# ---------------------------------------------------------
# DIAGRAM: STAGE 1 (ACD)
# ---------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot ACD Loss
ax1.plot(epochs, df_acd["Training Loss"], marker="o", color="#1f77b4", linewidth=2, label="Train Loss")
ax1.plot(epochs, df_acd["Validation Loss"], marker="s", color="#d62728", linewidth=2, linestyle="--", label="Val Loss")
ax1.set_title("Stage 1 (ACD): Training & Validation Loss", fontsize=12, fontweight="bold")
ax1.set_xlabel("Epoch", fontsize=11)
ax1.set_ylabel("Loss", fontsize=11)
ax1.set_xticks(epochs)
ax1.legend(frameon=True)
ax1.grid(True, alpha=0.3)

# Plot ACD Metrics
ax2.plot(epochs, df_acd["Acd Macro F1"], marker="o", color="#2ca02c", linewidth=2, label="Macro F1")
ax2.plot(epochs, df_acd["Acd Micro F1"], marker="^", color="#ff7f0e", linewidth=2, label="Micro F1")
ax2.plot(epochs, df_acd["Acd Accuracy"], marker="d", color="#9467bd", linewidth=2, linestyle=":", label="Accuracy")
ax2.set_title("Stage 1 (ACD): Performance Metrics", fontsize=12, fontweight="bold")
ax2.set_xlabel("Epoch", fontsize=11)
ax2.set_ylabel("Score", fontsize=11)
ax2.set_xticks(epochs)
ax2.legend(frameon=True)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ACD_METRICS_PER_EPOCH, dpi=300)
plt.close()
print(f"Successfully generated and saved {ACD_METRICS_PER_EPOCH}")

# ---------------------------------------------------------
# DIAGRAM: STAGE 2 (ACSA)
# ---------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot ACSA Loss
ax1.plot(epochs, df_acsa["Training Loss"], marker="o", color="#1f77b4", linewidth=2, label="Train Loss")
ax1.plot(epochs, df_acsa["Validation Loss"], marker="s", color="#d62728", linewidth=2, linestyle="--", label="Val Loss")
ax1.set_title("Stage 2 (ACSA): Training & Validation Loss", fontsize=12, fontweight="bold")
ax1.set_xlabel("Epoch", fontsize=11)
ax1.set_ylabel("Loss", fontsize=11)
ax1.set_xticks(epochs)
ax1.legend(frameon=True)
ax1.grid(True, alpha=0.3)

# Plot ACSA Metrics
ax2.plot(epochs, df_acsa["Acsa Macro F1"], marker="o", color="#2ca02c", linewidth=2, label="Macro F1")
ax2.plot(epochs, df_acsa["Acsa Micro F1"], marker="^", color="#ff7f0e", linewidth=2, label="Micro F1")
ax2.plot(epochs, df_acsa["Acsa Accuracy"], marker="d", color="#9467bd", linewidth=2, linestyle=":", label="Accuracy")
ax2.set_title("Stage 2 (ACSA): Performance Metrics", fontsize=12, fontweight="bold")
ax2.set_xlabel("Epoch", fontsize=11)
ax2.set_ylabel("Score", fontsize=11)
ax2.set_xticks(epochs)
ax2.legend(frameon=True)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ACSA_METRICS_PER_EPOCH, dpi=300)
plt.close()
print(f"Successfully generated and saved {ACSA_METRICS_PER_EPOCH}")

Successfully generated and saved acd_metrics_curves.png
Successfully generated and saved acsa_metrics_curves.png
